# Solace Producer Demo

This notebook demonstrates how to use the Solace producer to publish synthetic JSON messages to a local Solace PubSub+ broker running in Docker.

## Prerequisites

- Solace PubSub+ running in Docker (port 55554)
- The `solace-example` package installed (`pip install -e .`)

In [ ]:
from solace_producer import SolaceProducer
import json

## Configuration

Configure the connection to your Solace broker:

In [ ]:
# Solace connection settings (default for local Docker)
SOLACE_HOST = "tcp://localhost:55554"
SOLACE_VPN = "default"
SOLACE_USERNAME = "default"
SOLACE_PASSWORD = "default"
TOPIC = "synthetic/events"

## Generate a Sample Message

Let's see what a synthetic message looks like:

In [ ]:
# Create producer instance (not connected yet)
producer = SolaceProducer(
    host=SOLACE_HOST,
    vpn_name=SOLACE_VPN,
    username=SOLACE_USERNAME,
    password=SOLACE_PASSWORD,
    topic=TOPIC,
)

# Generate a sample message
sample_message = producer.generate_synthetic_message()
print(json.dumps(sample_message, indent=2))

## Publish Messages to Solace

In [ ]:
# Publish a single message
with SolaceProducer(
    host=SOLACE_HOST,
    vpn_name=SOLACE_VPN,
    username=SOLACE_USERNAME,
    password=SOLACE_PASSWORD,
    topic=TOPIC,
) as producer:
    message = producer.generate_synthetic_message()
    producer.publish_message(message)
    print(f"Published message with event_id: {message['event_id']}")

In [ ]:
# Publish multiple messages
NUM_MESSAGES = 100

with SolaceProducer(
    host=SOLACE_HOST,
    vpn_name=SOLACE_VPN,
    username=SOLACE_USERNAME,
    password=SOLACE_PASSWORD,
    topic=TOPIC,
) as producer:
    published = producer.produce_messages(
        count=NUM_MESSAGES,
        verbose=True,
    )
    print(f"\nTotal published: {published} messages")

## High Throughput Test

In [ ]:
import time

NUM_MESSAGES = 1000

with SolaceProducer(
    host=SOLACE_HOST,
    vpn_name=SOLACE_VPN,
    username=SOLACE_USERNAME,
    password=SOLACE_PASSWORD,
    topic=TOPIC,
) as producer:
    start = time.time()
    published = producer.produce_messages(
        count=NUM_MESSAGES,
        verbose=True,
    )
    elapsed = time.time() - start
    
    print(f"\n{'='*50}")
    print(f"Published {published:,} messages in {elapsed:.2f}s")
    print(f"Throughput: {published/elapsed:,.1f} msg/sec")

## Message Schema

The synthetic messages have the following structure:

```json
{
  "event_id": "uuid",
  "timestamp": "ISO-8601 datetime",
  "event_type": "user_login|purchase|page_view|click|signup",
  "user": {
    "user_id": "uuid",
    "username": "string",
    "email": "email",
    "ip_address": "ipv4"
  },
  "device": {
    "type": "mobile|desktop|tablet",
    "os": "iOS|Android|Windows|macOS|Linux",
    "browser": "Chrome|Firefox|Safari|Edge"
  },
  "location": {
    "country": "country code",
    "city": "city name",
    "latitude": float,
    "longitude": float
  },
  "metadata": {
    "session_id": "uuid",
    "page_url": "url",
    "referrer": "url or null",
    "value": float
  }
}
```